In [1]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

# Reasoning System - Graph Builder 

## Graph Builder

### Preliminaries 

***Imports***

In [2]:
# Python imports 
import random 
from collections import defaultdict
from itertools import islice
from pprint import pp

# PyTorch imports 
import torch 
from torch.utils.data import DataLoader

# HRI_GNN imports
from hri_gnn.config.const import SLOT_TABLE, MAX_SLOTS
from hri_gnn.utils.paths import get_output_path
from hri_gnn.datasets import raven_flat_collate
from hri_gnn.datasets import RavenDataset
from hri_gnn.models.cnn_encoder import RavenCnnEncoder
from hri_gnn.models import graph_builder_v2 as gb

***Group By Problems `Utils`***

In [3]:
def group_by_problems(batch):
	# Number of crops is the number of entities
    number_of_entities = batch[0].shape[0]
    
    # regroups entities by problem, the problem is identified by (batch_id, config_id, problem_id)
    problems = defaultdict(list)
    
    for i in range(number_of_entities):
        entity = {'id': i, 'panel_id':batch[3][i].item() ,'component_id':batch[4][i].item() , 'attributes':batch[1][i], 'bbox':batch[2][i]}
        key_problem = (batch[7][i].item(), batch[5][i].item(), batch[6][i].item())
        
        nested_tree = defaultdict(lambda: defaultdict(list))
        p_id = batch[3][i].item()
        c_id = batch[4][i].item()
        nested_tree[p_id][c_id].append(entity)
    
        problems[key_problem].append({p_id: dict(components) for p_id, components in nested_tree.items()})
    return dict(problems)

***Load the Data***

In [4]:
torch.manual_seed(92)
random.seed(92)

val_dataset_raven_100 = RavenDataset(
    dataset='raven-100',
    split='val',
    percentage_limit=0.1
)

val_dataloader = DataLoader(
    dataset=val_dataset_raven_100,
    batch_size=10,
    shuffle=False,
    collate_fn=raven_flat_collate
)

***Load the CNN Encoder Model***

In [5]:
# load the saved model 

output_path = get_output_path()

raven_encoder = RavenCnnEncoder()

raven_encoder.load_state_dict(torch.load(f=output_path/'models'/'raven_cnn_encoder_final.pth'))

<All keys matched successfully>

In [6]:
next(iter(raven_encoder.parameters())).device

device(type='cpu')

***what's in a batch ?..***

In [7]:
# grab the first batch 
batch = list(islice(val_dataloader, 0,1))[0]
print(f"The length of the batch: {len(batch)}")

The length of the batch: 8


In [8]:
count = 0
for tmp_batch in val_dataloader:
    count+= 1
    conf_prblm_batch = torch.stack([tmp_batch[5],tmp_batch[6], tmp_batch[7]]).unique(dim=-1)
    numb_prblm= conf_prblm_batch.shape[1]
    zipped = zip(conf_prblm_batch[0].tolist(), conf_prblm_batch[1].tolist(), conf_prblm_batch[2].tolist())
    problem_ids = [(i,j,k) for i,j,k in zipped] 
    print(f"batch number: {count}, problems count: {numb_prblm}, (cnfg_id, prblm_id, btc_id): {problem_ids}")
print(f"dataset length: {len(val_dataset_raven_100)}")
print(f"dataloader length: {count}")

batch number: 1, problems count: 10, (cnfg_id, prblm_id, btc_id): [(0, 7, 9), (0, 97, 7), (1, 36, 3), (2, 16, 6), (3, 56, 2), (4, 46, 8), (5, 47, 0), (6, 6, 1), (6, 16, 5), (6, 76, 4)]
batch number: 2, problems count: 4, (cnfg_id, prblm_id, btc_id): [(0, 46, 1), (0, 67, 0), (2, 26, 2), (5, 37, 3)]
dataset length: 14
dataloader length: 2


In [9]:
flat_crops                = batch[0] # crops               ,shape: [xxx, 1, 80, 80]
flat_attributes           = batch[1] # attributes          ,shape: [xxx, 4]
flat_bboxes               = batch[2] # bbox                ,shape: [xxx, 4] 
flat_panel_indices        = batch[3] # panel indices       ,shape: [xxx]
flat_component_indices    = batch[4] # component indices   ,shape: [xxx]
flat_config_indices       = batch[5] # config indices      ,shape: [xxx]
flat_problem_indices      = batch[6] # problems indices    ,shape: [xxx]
flat_batch_indices        = batch[7] # batch indices       ,shape: [xxx]

***Encode batch's entities (crops) `CNN Encoder - Forward pass`***

In [10]:
raven_encoder.eval()
with torch.inference_mode():
    model_output = raven_encoder(flat_crops)

pp(f"Model output dictionnary keys: {model_output.keys()}")

("Model output dictionnary keys: dict_keys(['latent_vector', "
 "'pred_type_logits', 'pred_size_logits', 'pred_color_logits', "
 "'pred_angle_logits'])")


### ***Graph Builder functions***

***group_entities_by_problem***

In [11]:
# indicies of entities grouped by problem
problems = gb.group_entities_by_problem(flat_config_indices,flat_problem_indices,flat_batch_indices)

pp(problems)

{(5, 47, 0): tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31]),
 (6, 6, 1): tensor([32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49,
        50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63]),
 (3, 56, 2): tensor([64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81,
        82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95]),
 (1, 36, 3): tensor([ 96,  97,  98,  99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109,
        110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121]),
 (6, 76, 4): tensor([122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135,
        136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149,
        150, 151, 152, 153]),
 (6, 16, 5): tensor([154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167,
        168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 

In [12]:
# Grab a problem 
problem_key , indices = next(((pk,pids) for pk,pids in problems.items() if pk == (4,46,8)))

print(f" - Problem: {problem_key}\n - indices: {indices}")

 - Problem: (4, 46, 8)
 - indices: tensor([285, 286, 287, 288, 289, 290, 291, 292, 293, 294, 295, 296, 297, 298,
        299, 300, 301, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311, 312,
        313, 314, 315, 316, 317, 318, 319, 320, 321, 322, 323, 324, 325, 326,
        327, 328, 329, 330, 331, 332, 333, 334, 335, 336, 337, 338, 339, 340,
        341])


***entities_slot_ids***

Using this slot table each entity have its own slot_id : 
```python
SLOT_TABLE = {
    (0,0):[[0.5, 0.5, 1, 1]],
    (1,0):[[0.25, 0.25, 0.5, 0.5], [0.25, 0.75, 0.5, 0.5], [0.75, 0.25, 0.5, 0.5], [0.75, 0.75, 0.5, 0.5]],
    (2,0):[[0.16, 0.16, 0.33, 0.33], [0.16, 0.5, 0.33, 0.33], [0.16, 0.83, 0.33, 0.33], [0.5, 0.16, 0.33, 0.33], [0.5, 0.5, 0.33, 0.33], [0.5, 0.83, 0.33, 0.33], [0.83, 0.16, 0.33, 0.33], [0.83, 0.5, 0.33, 0.33], [0.83, 0.83, 0.33, 0.33]],
    (3,0):[[0.5, 0.5, 1, 1]],
    (3,1):[[0.5, 0.5, 0.33, 0.33]],
    (4,0):[[0.5, 0.5, 1, 1]],
    (4,1):[[0.42, 0.42, 0.15, 0.15], [0.42, 0.58, 0.15, 0.15], [0.58, 0.42, 0.15, 0.15], [0.58, 0.58, 0.15, 0.15]],
    (5,0):[[0.5, 0.25, 0.5, 0.5]],
    (5,1):[[0.5, 0.75, 0.5, 0.5]],
    (6,0):[[0.25, 0.5, 0.5, 0.5]],
    (6,1):[[0.75, 0.5, 0.5, 0.5]]
}
```
key = (config_id, component_id) , value = Position attriute of the layout

In [13]:
slot_ids = gb.entities_slot_ids(indices, flat_bboxes, flat_config_indices, flat_component_indices,slot_table= SLOT_TABLE)
slot_ids

tensor([0, 3, 0, 1, 0, 0, 0, 1, 3, 0, 3, 2, 1, 0, 0, 1, 0, 0, 1, 2, 3, 0, 1, 2,
        0, 3, 0, 1, 2, 0, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 3, 1, 0, 0,
        2, 0, 1, 2, 3, 0, 1, 2, 3])

***build_entity_nodes***

Each problem have N entites will have N nodes each node have :
* Tensor with shape [96] = latent(64) + type_logits(6) + size_logits(6) + color_logits(10) + angle_logits(8)
* Metadata:
   - panel_id
   - component_id
   - config_id
   - slot_id
* id : local problem id mapped to its id in the batch 


In [14]:
entity_x , entity_meta, local_idx_map = gb.build_entity_nodes( indices, 
                                                                model_output, 
                                                                flat_panel_indices, 
                                                                flat_component_indices,
                                                                flat_config_indices, 
                                                                slot_ids)

print(f" * entity_x : tensor with shape:\t{entity_x.shape}\n")
print(f" * {entity_x.shape[0]} nodes (each entity of this problem is a is a node)\n")
print(f" * {entity_x.shape[1]} = latent(64) + type_logits(6) + size_logits(6) + color_logits(10) + angle_logits(8)\n")
print(" * Metadata dictionnary Keys: \n")
for key in entity_meta.keys():
    print(f"\t- '{key}'\t metadata is a tensor with shape {entity_meta[key].shape}")
print(f"\n * Local mapping ids:")
pp(f"{local_idx_map}")

 * entity_x : tensor with shape:	torch.Size([57, 94])

 * 57 nodes (each entity of this problem is a is a node)

 * 94 = latent(64) + type_logits(6) + size_logits(6) + color_logits(10) + angle_logits(8)

 * Metadata dictionnary Keys: 

	- 'panel_id'	 metadata is a tensor with shape torch.Size([57])
	- 'component_id'	 metadata is a tensor with shape torch.Size([57])
	- 'config_id'	 metadata is a tensor with shape torch.Size([57])
	- 'slot_id'	 metadata is a tensor with shape torch.Size([57])

 * Local mapping ids:
('{285: 0, 286: 1, 287: 2, 288: 3, 289: 4, 290: 5, 291: 6, 292: 7, 293: 8, '
 '294: 9, 295: 10, 296: 11, 297: 12, 298: 13, 299: 14, 300: 15, 301: 16, 302: '
 '17, 303: 18, 304: 19, 305: 20, 306: 21, 307: 22, 308: 23, 309: 24, 310: 25, '
 '311: 26, 312: 27, 313: 28, 314: 29, 315: 30, 316: 31, 317: 32, 318: 33, 319: '
 '34, 320: 35, 321: 36, 322: 37, 323: 38, 324: 39, 325: 40, 326: 41, 327: 42, '
 '328: 43, 329: 44, 330: 45, 331: 46, 332: 47, 333: 48, 334: 49, 335: 50, 336: '
 '

***build_intra_panel_entity_edges***

* Intra panel edges are between entities in the same component, having edges between entities with different components (expl. (Component: OUT)-E <-> E-(Component: IN)) have no semnatic rule oriented sense, positions signal will be captured by the 'slot_ids'
* Since bbox format [Cy, CX, W, H] Edges attributes are : 
  - delta_y  = destination(CY) - source(CY) 
  - delta_x = destination(CX) - source(CX)  

In [15]:
edge_indexes_intra, edgge_attrs_intra = gb.build_intra_panel_entity_edges(entity_meta, flat_bboxes, indices)

print(f" * Edge directions: {edge_indexes_intra.shape[0]}")
print(f" * Number of double directed intra edges in this problem: {int(edge_indexes_intra.shape[1]/edge_indexes_intra.shape[0])}")
print(f" * Edge attibutes Tensor shape: {edgge_attrs_intra.shape}")
print(f" \t* {edgge_attrs_intra.shape[0]} = {int(edge_indexes_intra.shape[1]/edge_indexes_intra.shape[0])} x {edge_indexes_intra.shape[0]}")
print(f" \t* {edgge_attrs_intra.shape[1]}:\n\t - delta_y: {edgge_attrs_intra[0][0]}\n\t - delta_x: {edgge_attrs_intra[0][1]}")

 * Edge directions: 2
 * Number of double directed intra edges in this problem: 39
 * Edge attibutes Tensor shape: torch.Size([78, 2])
 	* 78 = 39 x 2
 	* 2:
	 - delta_y: 0.0
	 - delta_x: -0.1599999964237213


***get_all_groups***

 * RAVEN rules are row-wise: 
  - Chi Zhang et al.,2019: A dataset for relational and analogical visual reasoning: "In the meantime, rules are applied row-wise"
  
  We will not include columns, it migh add unwanted noise to our model 
```python 
include_columns=False 
```
 * In RAVEN dataset panels from 0->15 , candidates are in [8,15]

 * Groups : all possible solution 
```python 
[0,1,2],
[3,4,5],
[6,7,X] => every X in range(8,16)  
```

In [41]:
include_columns= True
candidate_ids= range(8,16)

groups = gb.get_all_groups(candidate_ids, include_columns=include_columns)

pp(groups)

[[0, 1, 2],
 [3, 4, 5],
 [6, 7],
 [0, 3, 6],
 [1, 4, 7],
 [2, 5],
 [6, 8],
 [7, 8],
 [2, 8],
 [5, 8],
 [6, 9],
 [7, 9],
 [2, 9],
 [5, 9],
 [6, 10],
 [7, 10],
 [2, 10],
 [5, 10],
 [6, 11],
 [7, 11],
 [2, 11],
 [5, 11],
 [6, 12],
 [7, 12],
 [2, 12],
 [5, 12],
 [6, 13],
 [7, 13],
 [2, 13],
 [5, 13],
 [6, 14],
 [7, 14],
 [2, 14],
 [5, 14],
 [6, 15],
 [7, 15],
 [2, 15],
 [5, 15]]


***group_problem_entities_by_panel***

In [17]:
problem_entities_groupped_by_panel = gb.group_problem_entities_by_panel(entity_meta)
pp(problem_entities_groupped_by_panel)

{0: [0, 1],
 1: [2, 3, 4],
 2: [5, 6, 7, 8],
 3: [9, 10, 11, 12],
 4: [13, 14, 15],
 5: [16, 17, 18, 19, 20],
 6: [21, 22, 23],
 7: [24, 25],
 8: [26, 27, 28, 29, 30],
 9: [31, 32, 33, 34],
 10: [35, 36, 37, 38],
 11: [39, 40, 41, 42],
 12: [43, 44, 45, 46],
 13: [47, 48],
 14: [49, 50, 51, 52],
 15: [53, 54, 55, 56]}


***build_inter_panel_entity_edges***

* Inter panel edges are between entities having the same component id and same slot id but in different panels.
* Different panels in the same group of panels (i.e. row groups, like [0,1,2] or [6,7,X] for every X in range(8,16))
* Edge attributes: a tensor whith shape [number_of_edges x 31] 
* 31 = edge direction(1) + delta_logits(30) : 
  - edge direction possible values : 1 and  -1
  - delta logits : destination(type_logits, size_logits, color_logots, angle_logits) - source(type_logits, size_logits, color_logots, angle_logits)

In [18]:
edge_indexes_inter_panels_entities, edge_attrs_inter_panels_entities = gb.build_inter_panel_entity_edges(entity_meta, entity_x,
                                                                                                            problem_entities_groupped_by_panel, 
                                                                                                            groups)

print(f" * Edge directions: {edge_indexes_inter_panels_entities.shape[0]}")
print(f" * Number of double directed inter panel entity edges in this problem: {int(edge_indexes_inter_panels_entities.shape[1]/edge_indexes_inter_panels_entities.shape[0])}")
print(f" * Edge attibutes Tensor shape: {edge_attrs_inter_panels_entities.shape}")
print(f" \t* {edge_attrs_inter_panels_entities.shape[0]} = {int(edge_indexes_inter_panels_entities.shape[1]/edge_indexes_inter_panels_entities.shape[0])} x {edge_indexes_inter_panels_entities.shape[0]}")
print(f" \t* {edge_attrs_inter_panels_entities.shape[1]}:\n\t - direction: {edge_attrs_inter_panels_entities[0][0]}\n\t - delta_logits: (with shape:\t{edge_attrs_inter_panels_entities[0][1:].shape})\n\t {edge_attrs_inter_panels_entities[0][1:]}")

 * Edge directions: 2
 * Number of double directed inter panel entity edges in this problem: 60
 * Edge attibutes Tensor shape: torch.Size([120, 31])
 	* 120 = 60 x 2
 	* 31:
	 - direction: 1.0
	 - delta_logits: (with shape:	torch.Size([30]))
	 tensor([  5.5273,   4.8371,   3.8492,  -1.6182, -11.7007, -15.5248,   5.4444,
          3.8819,   2.0588,   3.3339,   1.9537,   3.5276,  -0.3049,  -0.9842,
          0.6417,   2.0223,   0.9963,   2.2913,   1.5852,   3.3746,   1.4848,
         -2.7633,   1.0070,   2.8647,  -0.7908,   6.0502,   3.1293,  -8.6602,
          2.3983,   5.4783])


***build_layout_nodes***

Layout nodes are scoped with (panel_id, component_id) which means for a problem having :
- 16 panels 
- nb_cmp : number of component (1 or 2 depends on the configuration)

will have 16 or 32 layout nodes
Each node have :

* Tensor with shape [number of panels (16 or 32) x 105 = 94 + 9 + 1 + 1], concatenation of:
  - pooled (using mean) entity nodes of this (panel, component) (94)
  - slot occumaptions mask : padded to max_slots in all configuartions (9)  
  - number of children (1)
  - component_id (1) (possible values : 0 ou 1)
* layout_key : (panel_id , component_id), Exemple (4,1)
* id : mapped id to layout_key = (panel_id , component_id), Exemple (4,1): 9 
* entities list : local problem entity's ids are grouped by layout_key = (panel_id, component_id), Exemple (4,1): [14, 15]

In [19]:
layout_x, layout_keys, layout_local_idx_map, local_entities_idx_grouped_by_layout_keys = gb.build_layout_nodes(entity_meta, 
                                                                                            entity_x,
                                                                                            max_slots=MAX_SLOTS)
print(f" * panel_x shape : panel_x.shape\n")
print(f" * panel keys :")
pp(f"{layout_keys}")
print(f"\n * Layout Local mapping:")
pp(f"{layout_local_idx_map}")
print(f"\n * local entities groupped by layout :")
pp(f"{local_entities_idx_grouped_by_layout_keys}")

 * panel_x shape : panel_x.shape

 * panel keys :
('[(0, 0), (0, 1), (1, 0), (1, 1), (2, 0), (2, 1), (3, 0), (3, 1), (4, 0), (4, '
 '1), (5, 0), (5, 1), (6, 0), (6, 1), (7, 0), (7, 1), (8, 0), (8, 1), (9, 0), '
 '(9, 1), (10, 0), (10, 1), (11, 0), (11, 1), (12, 0), (12, 1), (13, 0), (13, '
 '1), (14, 0), (14, 1), (15, 0), (15, 1)]')

 * Layout Local mapping:
('{(0, 0): 0, (0, 1): 1, (1, 0): 2, (1, 1): 3, (2, 0): 4, (2, 1): 5, (3, 0): '
 '6, (3, 1): 7, (4, 0): 8, (4, 1): 9, (5, 0): 10, (5, 1): 11, (6, 0): 12, (6, '
 '1): 13, (7, 0): 14, (7, 1): 15, (8, 0): 16, (8, 1): 17, (9, 0): 18, (9, 1): '
 '19, (10, 0): 20, (10, 1): 21, (11, 0): 22, (11, 1): 23, (12, 0): 24, (12, '
 '1): 25, (13, 0): 26, (13, 1): 27, (14, 0): 28, (14, 1): 29, (15, 0): 30, '
 '(15, 1): 31}')

 * local entities groupped by layout :
('{(0, 0): [0], (0, 1): [1], (1, 0): [2], (1, 1): [3, 4], (2, 0): [5], (2, 1): '
 '[6, 7, 8], (3, 0): [9], (3, 1): [10, 11, 12], (4, 0): [13], (4, 1): [14, '
 '15], (5, 0): [16], (5, 1): [

***build_inter_panel_layout_edges***

* Inter panel layout edges are between layouts having the same component id but in different panels.
* Different panels in the same group of panels (i.e. row groups, like [0,1,2] or [6,7,X] for every X in range(8,16))
* Edge attributes: a tensor whith shape [number_of_edges x 5] 
* 5 = edge direction(1) + delta_number(1) + n_common(1) + n_added(1) + n_removed(1) : 
  - edge direction possible values : 1 and  -1
  - delta_number : destination(number of entities in layout) - source(number of entities in layout)
  - n_common : number of common entities between layout source and destination (slot based)
  - n_added : number of added entities to destination (slot based)
  - n_removed : number of entites removed from destination (slot based)
    * slot_based means having same slot id 


In [20]:
edge_indexes_inter_panels_layout, edge_attrs_inter_panels_layout = gb.build_inter_panel_layout_edges(local_entities_idx_grouped_by_layout_keys, 
                                                                                                    layout_local_idx_map,
                                                                                                    entity_meta, 
                                                                                                    groups)

# edge_indexes_inter_panels_panels, edge_attrs_inter_panels_panels 
print(f" * Edge directions: {edge_indexes_inter_panels_layout.shape[0]}")
print(f" * Number of double directed layout inter panels edges in this problem: {int(edge_indexes_inter_panels_layout.shape[1]/edge_indexes_inter_panels_layout.shape[0])}")
print(f" * Edge attibutes Tensor shape: {edge_attrs_inter_panels_layout.shape}")
print(f" \t* {edge_attrs_inter_panels_layout.shape[0]} = {int(edge_indexes_inter_panels_layout.shape[1]/edge_indexes_inter_panels_layout.shape[0])} x {edge_indexes_inter_panels_layout.shape[0]}")
print(f" \t* {edge_attrs_inter_panels_layout.shape[1]}:\n\t - direction: {edge_attrs_inter_panels_layout[0][0]}\n\t - delta_number: {edge_attrs_inter_panels_layout[0][1]}\n\t - n_common: {edge_attrs_inter_panels_layout[0][2]}\n\t - n_added: {edge_attrs_inter_panels_layout[0][3]}\n\t - n_removed: {edge_attrs_inter_panels_layout[0][4]}")

 * Edge directions: 2
 * Number of double directed layout inter panels edges in this problem: 60
 * Edge attibutes Tensor shape: torch.Size([120, 5])
 	* 120 = 60 x 2
 	* 5:
	 - direction: 1.0
	 - delta_number: 0.0
	 - n_common: 1.0
	 - n_added: 0.0
	 - n_removed: 0.0


***build_belongs_to_edges***

* build edges : betwen entity-node and layout_node:
  - One direction entity-node --belongs_to--> layout-node
  - indexes are source (local entity indices) destination (local layout indices)
  
* Edge attributes is a simple tensor of ones with shape = [number of entities in that layout]  

In [21]:
edge_index_element_to_layout, edge_attr_element_to_layout = gb.build_belongs_to_edges(local_entities_idx_grouped_by_layout_keys,layout_local_idx_map)


print(f"Layout indices\n{edge_index_element_to_layout[0]}")
print(f"entity indices\n{edge_index_element_to_layout[1]}")
print(f"edge attibutes, a tensor with shape : {edge_attr_element_to_layout.shape} of values = 1")


Layout indices
tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35,
        36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53,
        54, 55, 56])
entity indices
tensor([ 0,  1,  2,  3,  3,  4,  5,  5,  5,  6,  7,  7,  7,  8,  9,  9, 10, 11,
        11, 11, 11, 12, 13, 13, 14, 15, 16, 17, 17, 17, 17, 18, 19, 19, 19, 20,
        21, 21, 21, 22, 23, 23, 23, 24, 25, 25, 25, 26, 27, 28, 29, 29, 29, 30,
        31, 31, 31])
edge attibutes, a tensor with shape : torch.Size([57, 1]) of values = 1


***build_contains_edges***

* build edges : betwen layout_node and entity-node :
  - One direction layout-node --contains--> entity-node
  - indexes are source (local layout indices) destination (local entity indices)
  
* Edge attributes is a simple tensor of ones with shape = [number of entities in that layout]  

In [22]:
edge_index_layout_to_element, edge_attr_layout_to_element = gb.build_contains_edges(local_entities_idx_grouped_by_layout_keys,layout_local_idx_map)

print(f"Layout indices\n{edge_index_layout_to_element[0]}")
print(f"entity indices\n{edge_index_layout_to_element[1]}")
print(f"edge attibutes, a tensor with shape : {edge_attr_layout_to_element.shape} of values = 1")

Layout indices
tensor([ 0,  1,  2,  3,  3,  4,  5,  5,  5,  6,  7,  7,  7,  8,  9,  9, 10, 11,
        11, 11, 11, 12, 13, 13, 14, 15, 16, 17, 17, 17, 17, 18, 19, 19, 19, 20,
        21, 21, 21, 22, 23, 23, 23, 24, 25, 25, 25, 26, 27, 28, 29, 29, 29, 30,
        31, 31, 31])
entity indices
tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35,
        36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53,
        54, 55, 56])
edge attibutes, a tensor with shape : torch.Size([57, 1]) of values = 1


***build_hetero_data_for_problem***

In [23]:
data = gb.build_hetero_data_for_problem(indices, 
                                model_output, 
                                flat_bboxes, 
                                flat_panel_indices,
                                flat_component_indices, 
                                flat_config_indices, 
                                slot_table=SLOT_TABLE, 
                                max_slots=MAX_SLOTS, 
                                candidate_ids= range(8,16),
                                include_columns=False)
pp(data.keys())

['edge_index',
 'layout_local_idx_map',
 'entity_local_idx_map',
 'layout_keys',
 'entity_meta',
 'x',
 'local_entities_idx_grouped_by_layout_keys',
 'edge_attr']


***build_hetero_data_batch***

In [42]:
from hri_gnn.config.const import Configuration
hetero_data_dict = gb.build_hetero_data_batch(batch, raven_encoder, include_columns= True, candidate_ids=range(8,16), device= None)

print(f"number of problems: {len(hetero_data_dict)}")
problem_key =  (4, 46, 8)  #(0, 7, 9)
problem_hetero_data = hetero_data_dict[problem_key]
print("Exemple (first problem):\n")
print(f" - problem key: {problem_key}, CONFIG: {Configuration.all()[problem_key[0]]}")
print(f" - problem data:")
print(f"    - Type: {type(problem_hetero_data)}")
print(f"    - properties: ")
print(f"       - nodes_count:")
for node_type, node_count in problem_hetero_data['nodes_count'].items():
    print(f"               - {node_type}: {node_count} , \tx dimension: {problem_hetero_data[node_type].x.shape[1]}")
print(f"       - edges_count:")
for edge_type, edge_count in problem_hetero_data['edges_count'].items():
    print(f"               - {edge_type}: {edge_count} , \tattr dimension: {problem_hetero_data[edge_type].edge_attr.shape[1]}")


number of problems: 10
Exemple (first problem):

 - problem key: (4, 46, 8), CONFIG: in_distribute_four_out_center_single
 - problem data:
    - Type: <class 'torch_geometric.data.hetero_data.HeteroData'>
    - properties: 
       - nodes_count:
               - entity: 57 , 	x dimension: 86
               - layout: 32 , 	x dimension: 97
       - edges_count:
               - ('entity', 'intra_panel', 'entity'): 78 , 	attr dimension: 2
               - ('entity', 'inter_panel_entity', 'entity'): 246 , 	attr dimension: 23
               - ('layout', 'inter_panel_layout', 'layout'): 184 , 	attr dimension: 5
               - ('entity', 'belongs_to', 'layout'): 57 , 	attr dimension: 1
               - ('layout', 'contains', 'entity'): 57 , 	attr dimension: 1
